# ¿Dónde se decidió la investidura? Escaños en juego en noviembre de 2019

En las elecciones generales de noviembre de 2019, ¿en qué provincias se quedó un partido a pocos votos de arrebatar el último escaño al bloque contrario?

1. Se reparten los escaños de cada provincia con la ley d'Hondt.
2. En la última asignación de cada provincia se compara el partido que se lleva el escaño con el que se queda más cerca. Solo interesan los casos en los que son de bloques distintos respecto a la investidura de Pedro Sánchez.
3. En las provincias más ajustadas se comprueba si ese segundo partido habría ganado el escaño con los votos que perdió desde abril de 2019, municipio a municipio.

**Datos necesarios** (ver [`data/README.md`](../data/README.md)):

- `raw/resultados_electorales/congreso_por_provincia/PROV_02_201911_1.xlsx`: resultados por provincia de noviembre de 2019.
- `raw/escaños/escaños_2019.xlsx`: escaños por provincia (`codigo_de_provincia`, `nombre_de_provincia`, `escaños`).
- `staging/elecciones_generales_completas.csv`: resultados por municipio; los genera el notebook 01.
- `raw/mapas/mapa_provincias.json` y `raw/mapas/zaragoza_municipios.geojson`.

El resultado de la tabla d'Hondt se publicó en [`resultados/dhondt_last_line.csv`](../resultados/dhondt_last_line.csv).

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd
import plotly.express as px

sys.path.insert(0, str(Path.cwd().parent))  # permite importar el paquete politicai desde notebooks/
from politicai.config import RAW_DIR, STAGING_DIR, publicar_en_chart_studio
from politicai.electoral import COLUMNAS_CLAVE_PROVINCIA, calculate_dhondt, columnas_partidos
from politicai.limpieza import clean_column_names

pd.set_option("display.max_columns", None)

In [ ]:
ELECCIONES_PROVINCIA_PATH = RAW_DIR / "resultados_electorales" / "congreso_por_provincia" / "PROV_02_201911_1.xlsx"
ELECCIONES_MUNICIPIO_PATH = STAGING_DIR / "elecciones_generales_completas.csv"
ESCANIOS_PATH = RAW_DIR / "escaños" / "escaños_2019.xlsx"
GEOJSON_PROV_PATH = RAW_DIR / "mapas" / "mapa_provincias.json"
GEOJSON_ZARAGOZA_PATH = RAW_DIR / "mapas" / "zaragoza_municipios.geojson"

Bloques según su voto en la investidura de enero de 2020:

In [ ]:
coalicion_a_favor = ["psoe", "podemos_iu", "eaj_pnv", "mas_pais_equo", "bng", "teruel_existe!",
                     "ecp_guanyem_el_canvi", "podemos_eu", "mes_compromis", "mas_pais", "m_pais_cha_equo"]
coalicion_en_contra = ["pp", "vox", "cs", "jxcat_junts", "cup_pr", "upn", "cca_pnc_nc", "Foro Asturias", "prc", "na+"]
coalicion_abstencion = ["erc_sobiranistes", "eh_bildu"]

In [ ]:
def limpiar_elecciones_provincia(df):
    """Aplana la cabecera de dos niveles del Excel por provincia y deja solo los votos de cada partido."""
    df = df.drop(columns=[c for c in df.columns if "Diputados" in c])
    # Las columnas sin título en el primer nivel aparecen como "Unnamed: ..."
    df.columns = [" ".join(("" if "Unnamed" in nivel0 else nivel0, nivel1)).strip() for nivel0, nivel1 in df.columns]
    df = clean_column_names(df)
    df.columns = [col.replace("_votos", "") for col in df.columns]

    columnas_texto = ["nombre_de_comunidad", "codigo_de_provincia", "nombre_de_provincia"]
    columnas_numericas = [c for c in df.columns if c not in columnas_texto]
    df[columnas_numericas] = df[columnas_numericas].apply(pd.to_numeric)
    return df, columnas_partidos(df, COLUMNAS_CLAVE_PROVINCIA)


def votos_perdidos(df_elecciones_municipio, codigo_prov, df_prov_diff):
    """Votos que perdió, entre abril y noviembre de 2019, el partido que se quedó a las puertas del último escaño.

    Solo se suman los municipios donde ese partido bajó. Devuelve el total y la
    tabla por municipio.
    """
    fila = df_prov_diff[df_prov_diff["codigo_de_provincia"] == str(codigo_prov).zfill(2)]
    partido = fila["partido_silver"].iloc[0].replace("cs", "ciudadanos")  # el fichero por municipio usa "ciudadanos"

    df_prov = df_elecciones_municipio[
        (df_elecciones_municipio["codigo_de_provincia"] == codigo_prov)
        & (df_elecciones_municipio["yearmonth"].astype(str).isin(["201904", "201911"]))
    ]
    pivot = df_prov.pivot(index=["codigo_de_municipio", "nombre_de_municipio"], columns="yearmonth", values=partido)
    pivot.columns = [f"{partido}_{c}" for c in pivot.columns]
    pivot[f"perdidos_{partido}"] = pivot[f"{partido}_201904"] - pivot[f"{partido}_201911"]
    pivot = pivot[pivot[f"perdidos_{partido}"] > 0]
    return pivot[f"perdidos_{partido}"].sum(), pivot

## Datos

In [ ]:
df_elecciones_municipio = pd.read_csv(ELECCIONES_MUNICIPIO_PATH, sep="|", dtype={"CODIGOINE": str}, index_col=0)
party_columns_municipio = columnas_partidos(df_elecciones_municipio)
df_elecciones_municipio[party_columns_municipio] = df_elecciones_municipio[party_columns_municipio].apply(pd.to_numeric)
df_elecciones_municipio.shape

In [ ]:
df_provincia_raw = pd.read_excel(ELECCIONES_PROVINCIA_PATH, header=[0, 1], skiprows=4)
df_provincia, party_columns_prov = limpiar_elecciones_provincia(df_provincia_raw)
df_provincia.shape

In [ ]:
df_escanios = pd.read_excel(ESCANIOS_PATH)
df_escanios.head()

## Reparto d'Hondt por provincia

In [ ]:
tablas_dhondt = {}
for _, row in df_escanios.iterrows():
    df_provincia_row = df_provincia[df_provincia["codigo_de_provincia"] == row["codigo_de_provincia"]]
    tabla, _ = calculate_dhondt(df_provincia_row, party_columns_prov, n_seats=row["escaños"])
    tablas_dhondt[row["codigo_de_provincia"]] = tabla

En la última fila de cada tabla, el partido con el cociente más alto (`gold`) se lleva el último escaño y el siguiente (`silver`) es el que se queda más cerca. `diferencia` es el cociente de los partidos a favor de la investidura menos el de los que están en contra: si es positiva, el escaño fue para el bloque a favor y el bloque en contra necesitaba aproximadamente esa cantidad de votos más.

In [ ]:
filas = []
for _, row in df_escanios.iterrows():
    last_line = tablas_dhondt[row["codigo_de_provincia"]].tail(1)
    gold = last_line.idxmax(axis=1).iloc[0]
    silver = last_line.drop(columns=gold).idxmax(axis=1).iloc[0]
    cocientes = last_line[[gold, silver]].iloc[0]
    a_favor = cocientes[[p for p in cocientes.index if p in coalicion_a_favor]].sum()
    en_contra = cocientes[[p for p in cocientes.index if p in coalicion_en_contra]].sum()
    filas.append({
        "codigo_de_provincia": str(row["codigo_de_provincia"]).zfill(2),
        "nombre_de_provincia": row["nombre_de_provincia"].strip(),
        "diferencia": a_favor - en_contra,
        "partido_gold": gold,
        "partido_silver": silver,
    })
df_prov_diff = pd.DataFrame(filas)

In [ ]:
# Provincias donde el último escaño fue al bloque a favor y el bloque en contra estuvo más cerca de quitárselo.
df_prov_best_candidates = df_prov_diff[
    (df_prov_diff["diferencia"] > 0) & ~df_prov_diff["partido_silver"].isin(coalicion_a_favor)
].sort_values("diferencia")
df_prov_best_candidates

In [ ]:
with open(GEOJSON_PROV_PATH) as geofile:
    geojson = json.load(geofile)

fig_map_provincias = px.choropleth_map(
    data_frame=df_prov_best_candidates, geojson=geojson,
    locations="codigo_de_provincia", featureidkey="properties.Codigo",
    color="diferencia", color_continuous_scale="Viridis_r",
    hover_data=["nombre_de_provincia", "diferencia", "partido_silver"],
    map_style="carto-positron", zoom=3.5, center={"lat": 36, "lon": -7},
    title="Votos restantes para conseguir el último escaño. Elecciones nov. 2019",
)
fig_map_provincias.show()
# publicar_en_chart_studio(fig_map_provincias, "votos_restantes_ultimo_escaño_nov_2019")

## ¿Habrían bastado los votos perdidos desde abril?

Para cada provincia ajustada se suman los votos que el partido `silver` perdió entre abril y noviembre de 2019, solo en los municipios donde bajó.

In [ ]:
PROVINCIAS_A_REVISAR = {"Cuenca": 16, "Huesca": 22, "Girona": 17, "Castellón": 12, "Albacete": 2, "Zaragoza": 50}

perdidos = {}
for nombre, codigo in PROVINCIAS_A_REVISAR.items():
    total, tabla = votos_perdidos(df_elecciones_municipio, codigo, df_prov_diff)
    perdidos[nombre] = tabla
    print(f"{nombre}: {total:.0f} votos perdidos")

Conclusiones:

| Provincia | Partido | Votos necesarios | Votos perdidos desde abril |
|---|---|---|---|
| Cuenca | Vox | 227 | 165 |
| Huesca | Vox | 1.813 | 83 |
| Girona | CUP | 2.007 | 0 (no se presentó en abril) |
| Castellón | PP | 4.163 | ver salida de la celda anterior* |
| Albacete | PP | 5.507 | 491 |
| Zaragoza | Ciudadanos | 5.774 | 68.485 |

Solo en **Zaragoza** los votos que perdió Ciudadanos desde abril habrían bastado para ganar el último escaño.

\* La versión original de este notebook calculaba Castellón con el código de provincia de Huesca (22) en lugar del suyo (12), así que la cifra publicada (83) no era la de Castellón.

## Zaragoza: votos perdidos por Ciudadanos por municipio

In [ ]:
with open(GEOJSON_ZARAGOZA_PATH) as geofile:
    geojson_zaragoza = json.load(geofile)

df_perdidos_zaragoza = perdidos["Zaragoza"].reset_index()
df_perdidos_zaragoza["CODIGOINE"] = "50" + df_perdidos_zaragoza["codigo_de_municipio"].astype(str).str.zfill(3)
CODIGO_ZARAGOZA_CAPITAL = 297


def mapa_zaragoza(datos, titulo, zoom=6):
    return px.choropleth_map(
        data_frame=datos, geojson=geojson_zaragoza,
        locations="CODIGOINE", featureidkey="properties.CODIGOINE",
        color="perdidos_ciudadanos", color_continuous_scale="Viridis",
        hover_data=["nombre_de_municipio", "perdidos_ciudadanos"],
        map_style="carto-positron", zoom=zoom, center={"lat": 41.8, "lon": -1},
        title=titulo,
    )

In [ ]:
fig_zaragoza = mapa_zaragoza(df_perdidos_zaragoza, "Votos perdidos por Ciudadanos en Zaragoza entre 2019-04 y 2019-11")
fig_zaragoza.show()

La capital concentra la mayoría de los votos, así que se repite el mapa sin ella:

In [ ]:
df_sin_capital = df_perdidos_zaragoza[df_perdidos_zaragoza["codigo_de_municipio"] != CODIGO_ZARAGOZA_CAPITAL]
fig_zaragoza_sin_capital = mapa_zaragoza(
    df_sin_capital, "Votos perdidos por Ciudadanos en Zaragoza entre 2019-04 y 2019-11 sin la capital"
)
fig_zaragoza_sin_capital.show()

Los seis municipios (sin contar la capital) donde Ciudadanos perdió más votos:

In [ ]:
df_destacados = df_sin_capital.nlargest(6, "perdidos_ciudadanos")
print(df_destacados["nombre_de_municipio"].to_list())
print("Total:", df_destacados["perdidos_ciudadanos"].sum())

In [ ]:
fig_zaragoza_destacados = mapa_zaragoza(
    df_destacados, "Votos perdidos por Ciudadanos en Zaragoza entre 2019-04 y 2019-11. Municipios destacados", zoom=6.5
)
fig_zaragoza_destacados.show()